# APL-Diffusion — Section 6.1 Experiment

**Paper:** Jin, Xu, Yang (2025) — *Adaptive Partitioning and Learning for Stochastic Control of Diffusion Processes*, arXiv:2512.14991

This notebook imports `apl_diffusion_full.py` as a library and runs each step of the experiment independently. Nothing executes on import — every cell is opt-in.

**File must be in the same directory as this notebook.**

---
### Structure
| Cell | What it does |
|------|-------------|
| 1 | Imports and configuration |
| 2 | Bellman solver → V\*(x₀) |
| 3 | MC constant-policy lower bound |
| 4 | Run APL-Diffusion experiments |
| 5 | Partition heatmap |
| 6 | Regret plots + slope summary |

## Cell 1 — Imports and configuration

In [ ]:
import time
import numpy as np

from apl_diffusion_full import (
    AdaDiffEnvironment,
    APLDiffusion,
    Experiment,
    BellmanSolverScalar,
    compute_vstar_gh,
    compute_best_constant,
    run_experiments,
    cumulative_regret,
    regret_slope,
    plot_all,
    plot_partition_heatmap,
    N_EPS,
    EP_LEN,
    STARTING_STATE,
)

# ── Solver resolution ────────────────────────────────────────────────────────
# Peak RAM = n_state * n_action * n_quad * 24 bytes (three float64 tensors)
# 8 GB machine:  use 801 / 401 / 61   → 0.47 GB  (safe)
# 16 GB machine: use 1601 / 801 / 121 → 3.70 GB  (higher accuracy)
# DO NOT use 1601/801/121 on an 8 GB machine — it will crash the kernel.
N_STATE   = 801    # state grid points
N_ACTION  = 401    # action grid points
N_QUAD    = 61     # Gauss-Hermite quadrature points

N_EXPERIMENTS = 20   # parallel APL-Diffusion runs
FIT_START     = 1000 # episode index for log-log slope fit

print(f'Library loaded. N_EPS={N_EPS}, EP_LEN={EP_LEN}, x0={STARTING_STATE}')
print(f'Solver resolution: {N_STATE} states x {N_ACTION} actions x {N_QUAD} quad pts')
peak_gb = N_STATE * N_ACTION * N_QUAD * 24 / 1e9
print(f'Peak RAM for solver: ~{peak_gb:.2f} GB')


## Cell 2 — Bellman solver: V\*(x₀)

Runs backward induction with vectorised Gauss-Hermite quadrature.
This is the reference line for the regret plots.

**Runtime:** ~2 min on a 2017 MacBook Air at full resolution.

In [ ]:
# BellmanSolverScalar is the fast vectorised solver.
# compute_vstar_gh() now delegates here, but calling directly is cleaner.
t0 = time.perf_counter()

solver    = BellmanSolverScalar(ep_len=EP_LEN,
                                n_state=N_STATE,
                                n_action=N_ACTION,
                                n_quad=N_QUAD)
solver.solve()
v_star_gh = solver.get_value(STARTING_STATE, h=0)

print(f'V*(x0={STARTING_STATE}) = {v_star_gh:.6f}')
print(f'Solver time: {time.perf_counter() - t0:.1f}s')


## Cell 3 — MC constant-policy lower bound

Computes the value of the best *constant* action via Monte Carlo.
This is a lower bound on V\* because the true optimal policy is state-dependent.
Useful as a sanity check: the gap shows how much the adaptive policy gains.

In [ ]:
t0 = time.perf_counter()

v_star_mc, best_action = compute_best_constant(
    n_actions = 201,
    n_mc      = 50_000,
    x0        = STARTING_STATE,
)

print(f"Best constant action:    a* = {best_action:.2f}")
print(f"V^const(x0)            = {v_star_mc:.6f}")
print(f"V*(x0) [GH solver]     = {v_star_gh:.6f}")
print(f"Gap (state-dep. gain)  = {v_star_gh - v_star_mc:.4f}")
print(f"Time: {time.perf_counter() - t0:.1f}s")

## Cell 4 — Run APL-Diffusion experiments

Runs `N_EXPERIMENTS` independent APL-Diffusion agents in parallel.
Returns a `(N_EPS, N_EXPERIMENTS)` matrix of episode rewards.

**Runtime:** ~2 min on a 2017 MacBook Air for N_EXPERIMENTS=20.

In [ ]:
t0 = time.perf_counter()

vpi_matrix = run_experiments(n=N_EXPERIMENTS)   # shape (N_EPS, N_EXPERIMENTS)

mean_final = vpi_matrix[-100:].mean()           # avg reward over last 100 episodes
print(f"{N_EXPERIMENTS} experiments x {N_EPS} episodes complete [{time.perf_counter()-t0:.1f}s]")
print(f"Mean episode reward (last 100 eps): {mean_final:.2f}")
print(f"V*(x0) for reference:               {v_star_gh:.2f}")

## Cell 5 — Partition heatmap

Trains one agent and plots the Q-value heatmap of its adaptive partition
at the final timestep — reproduces Figure 2 from the paper.

In [ ]:
env_vis   = AdaDiffEnvironment()
agent_vis = APLDiffusion(flag=True)
Experiment(env_vis, agent_vis, n_eps=N_EPS, seed=123).run()

plot_partition_heatmap(agent_vis, timestep=EP_LEN - 1)

## Cell 6 — Regret plots and slope

Produces the three-panel figure:
- **(a)** Learning curve with ±1σ band
- **(b)** Log-log cumulative regret vs episode — slope = regret exponent α
- **(c)** Slope comparison against theoretical worst-case bound (3/4)

The slope against the GH solver is the number comparable to Figure 3(b) of the paper (reported as ≈ 0.69).

In [ ]:
slopes = plot_all(
    vpi_matrix = vpi_matrix,
    v_star_gh  = v_star_gh,
    v_star_mc  = v_star_mc,
    fit_start  = FIT_START,
)

print("\n── Regret slope summary ─────────────────────────────────")
for name, sl in slopes.items():
    tag = "  ← compare with paper" if "GH" in name else ""
    print(f"  {name:<28s}  α = {sl:.4f}{tag}")
print(f"  Theoretical worst-case bound     α = 0.7500  (Theorem 5.19)")
print(f"  Paper Figure 3(b) reports        α ≈ 0.6900")

gh_slope = slopes.get('GH solver (paper ref)', float('nan'))
status = "✓ sublinear regret confirmed" if gh_slope < 0.75 else "✗ slope >= 0.75"
print(f"\n  {status}  (slope = {gh_slope:.4f})")

## Optional — manual regret inspection

In [ ]:
# Compute regret arrays directly if you want to inspect or replot manually
cum_mean, cum_lo, cum_hi = cumulative_regret(vpi_matrix, v_star_gh)
slope, intercept, r2     = regret_slope(cum_mean, fit_start=FIT_START)

print(f"Cumulative regret at K={N_EPS}: {cum_mean[-1]:.1f}")
print(f"Log-log slope α = {slope:.4f}  (R² = {r2:.4f})")